In [3]:
import pandas as pd
import sqlite3

conn = sqlite3.connect(':memory:')

pd.DataFrame({
    'id':         [1, 2, 3, 4, 5, 6],
    'name':       ['a', 'b', 'c', 'd', 'e', 'f'],
    'department': ['engineering', 'marketing', 'engineering', 'hr', 'marketing', 'engineering'],
    'salary':     [95000, 45000, 50000, 79000, 44000, 90000],
    'age':        [45, 22, 27, 29, 19, 31]
}).to_sql('employees', conn, index=False, if_exists='replace')

print("Ready!")

Ready!


In [5]:
result = pd.read_sql_query("""
    SELECT name, department, salary, 
        AVG(salary) OVER (PARTITION BY department) as dept_avg,
        salary - AVG(salary) OVER (PARTITION BY department) as diff_from_avg
    FROM employees
    ORDER BY department, salary DESC
""", conn)
print(result)

  name   department  salary      dept_avg  diff_from_avg
0    a  engineering   95000  78333.333333   16666.666667
1    f  engineering   90000  78333.333333   11666.666667
2    c  engineering   50000  78333.333333  -28333.333333
3    d           hr   79000  79000.000000       0.000000
4    b    marketing   45000  44500.000000     500.000000
5    e    marketing   44000  44500.000000    -500.000000


In [8]:
result = pd.read_sql_query("""
    SElECT name, department, salary,
        RANK() OVER (PARTITION BY department ORDER BY salary DESC) as dept_rank,
        RANK() OVER (ORDER BY salary DESC) as company_rank
    FROM employees
""", conn)
print(result)

  name   department  salary  dept_rank  company_rank
0    a  engineering   95000          1             1
1    f  engineering   90000          2             2
2    c  engineering   50000          3             4
3    d           hr   79000          1             3
4    b    marketing   45000          1             5
5    e    marketing   44000          2             6


In [10]:
result = pd.read_sql_query("""
    SELECT name, salary,
        LAG(salary) OVER (ORDER BY salary DESC) as prev_salary,
        salary - LAG(salary) OVER (ORDER BY salary DESC) as gap
    FROM employees
""", conn)
print(result)

  name  salary  prev_salary      gap
0    a   95000          NaN      NaN
1    f   90000      95000.0  -5000.0
2    d   79000      90000.0 -11000.0
3    c   50000      79000.0 -29000.0
4    b   45000      50000.0  -5000.0
5    e   44000      45000.0  -1000.0
